# LLM Prompting & Transformer Fine-Tuning for Text Classification

A comparative NLP study benchmarking four classification approaches on a Tang Dynasty poetry dataset.

**Approaches compared:**
- Logistic Regression (bag-of-words baseline)
- Fine-tuned Transformer (`all-MiniLM-L6-v2`)
- Zero-shot LLM (`gemma3:1b-it-qat` via Ollama)
- Few-shot LLM (dynamic example injection, k={1,3,5})

The project extends into **generative AI**: few-shot poem generation evaluated by the fine-tuned classifier.

**Dataset:** [Tang Dynasty Poetry](https://github.com/Leslie-Wong-H/Chinese-Poetry-Bilingual) — 97 bilingual poems, labeled nature vs. other  
**Author:** Akinwale Agesin

## Imports

All imports are declared in the cell below.

> **Ollama required for LLM sections.** The zero-shot and few-shot classification sections use [Ollama](https://ollama.com) to serve `gemma3:1b-it-qat` locally via an OpenAI-compatible API. To run those sections:
> ```bash
> ollama pull gemma3:1b-it-qat
> ollama serve
> ```
> The transformer fine-tuning and logistic regression sections run without Ollama.

In [48]:
import numpy as np
import pandas as pd
import torch
from openai import OpenAI

from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments
)

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, f1_score
from sklearn.metrics import precision_score, recall_score, accuracy_score

import re
from tqdm.notebook import tqdm

from typing import Optional, Sequence, Tuple

## 1. Data Loading and Preprocessing

### 1.1 Load and Format Data

The dataset is a bilingual collection of 97 Tang Dynasty poems from [Leslie Wong's Chinese Poetry repository](https://github.com/Leslie-Wong-H/Chinese-Poetry-Bilingual). This project uses the English translations.

Each poem has overlapping subject tags. We create a binary label: `1` if any tag relates to nature (landscape, seasons, moon, etc.), `0` otherwise.

In [49]:
# corpus source url
poems_json = 'https://raw.githubusercontent.com/Leslie-Wong-H/Chinese-Poetry-Bilingual/master/Tang/Tang.json'

# tags that indicate nature poems
nature_tags = set([
    'landscape',
    'spring',
    'mountain',
    'night',
    'windy',
    'summer',
    'winter',
    'autumn',
    'moon'
])

df = pd.read_json(poems_json)
df.set_index('index', inplace=True)

dfs_en, dfs_zh = [],[]

for idx in df.index:
  entry = df.loc[idx, 'English']
  entry['content'] = ' '.join(entry['content'])
  dfs_en.append(pd.DataFrame(entry, index=[idx]))
  entry = df.loc[idx, 'Chinese']
  entry['content'] = ' '.join(entry['content'])
  dfs_zh.append(pd.DataFrame(entry, index=[idx]))
en = pd.concat(dfs_en)
en.columns = ['en_'+i for i in en.columns]
zh = pd.concat(dfs_zh)
zh.columns = ['zh_'+i for i in zh.columns]
poems = en.join(zh).join(df['tags'])
poems.sample(3)

,en_title,en_dynasty,en_author,en_content,zh_title,zh_dynasty,zh_author,zh_content,tags
1-0063,Hard is the Road to Shu,Tang,Li Bai,Oho! Behold! How steep! How high! The road to ...,蜀道难,唐,李白,噫吁嚱， 危乎高哉！ 蜀道之难， 难于上青天！ 蚕丛及鱼凫， 开国何茫然！ 尔来四万八千岁，...,"[mountain, history, frustration]"
1-0041,A Fisherman,Tang,Liu Zongyuan,Under western cliff a fisherman passes the nig...,渔翁,唐,柳宗元,渔翁夜傍西岩宿， 晓汲清湘燃楚竹。 烟销日出不见人， 欸乃一声山水绿。 回看天际下中流， 岩...,"[night, silence]"
1-0094,The Lake in Spring,Tang,Bai Juyi,What a charming picture when spring comes to t...,春题湖上,唐,白居易,湖上春来似画图， 乱峰围绕水平铺。 松排山面千重翠， 月点波心一颗珠。 碧毯线头抽早稻， 青...,"[landscape, spring]"


Create a `"nature"` column: `1` if any of the `nature_tags` appear in a poem's tags, `0` otherwise.

In [50]:
poems["nature"] = poems["tags"].apply(
    lambda tags: int(any(tag in nature_tags for tag in tags))
)
poems["nature"].value_counts()

nature
0    61
1    36
Name: count, dtype: int64

In [51]:
# Get the percentage of poems that have nature tags
nature_pct = poems["nature"].mean() * 100
print(f"Percentage of poems that are nature-themed: {nature_pct:.2f}%")

Percentage of poems that are nature-themed: 37.11%


Create an `"en_formatted"` column concatenating title and content: `"{en_title}\n\n{en_content}"`.

In [52]:
poems["en_formatted"] = (
    poems["en_title"].astype(str)
    + "\n\n"
    + poems["en_content"].astype(str)
)


In [53]:
# Print the first poem in the dataframe
print(poems["en_formatted"].iloc[0])

Home-Coming

Oh, I return to the homeland I left while young, Thinner has grown my hair, though I speak the same tongue. My children, whom I meet, do not know who am I, "Where are you from, dear sir?" they ask with beaming eyes.


### 1.2 Train / Test Split

Split into train (80%) and test (20%) using stratified sampling to preserve class distribution.

In [54]:
X = poems["en_formatted"].tolist()
y = poems["nature"].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)


## 2. Classification

### 2.1 Baseline — Logistic Regression

`CountVectorizer` (max 5,000 features) + `LogisticRegression` trained on the training set. This establishes a performance floor against which the transformer and LLM approaches are compared.

In [55]:
# Fit your CountVectorizer and your LR classifier here
vectorizer = CountVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

lr_clf = LogisticRegression(max_iter=1000)
lr_clf.fit(X_train_vec, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


Evaluate precision, recall, and F1 on train and test sets.

In [56]:
y_train_pred = lr_clf.predict(X_train_vec)
y_test_pred = lr_clf.predict(X_test_vec)

print("Train classification report:")
print(classification_report(y_train, y_train_pred))

print("\nTest classification report:")
print(classification_report(y_test, y_test_pred))

train_f1 = f1_score(y_train, y_train_pred)
test_f1 = f1_score(y_test, y_test_pred)

print(f"Train F1: {train_f1:.3f}")
print(f"Test F1: {test_f1:.3f}")

Train classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        48
           1       1.00      1.00      1.00        29

    accuracy                           1.00        77
   macro avg       1.00      1.00      1.00        77
weighted avg       1.00      1.00      1.00        77


Test classification report:
              precision    recall  f1-score   support

           0       0.79      0.85      0.81        13
           1       0.67      0.57      0.62         7

    accuracy                           0.75        20
   macro avg       0.73      0.71      0.72        20
weighted avg       0.74      0.75      0.75        20

Train F1: 1.000
Test F1: 0.615


### 2.2 Fine-Tuned Transformer — `all-MiniLM-L6-v2`

Fine-tune a pretrained transformer for binary sequence classification. Despite being a small model, fine-tuning on even a modest labeled dataset significantly outperforms zero/few-shot LLM prompting on this domain-specific task.

Load the pretrained model and tokenizer.

In [57]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"

# Load the pretrained model.
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# get the model's max sequence length from your tokenizer, then print it
max_length = tokenizer.model_max_length 
print(max_length)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/all-MiniLM-L6-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


512


Tokenize train and test poems with padding and truncation to the model's max sequence length.

In [58]:
train_encodings = tokenizer(
    X_train,
    truncation=True,
    padding=True,
    max_length=max_length,
)

test_encodings = tokenizer(
    X_test,
    truncation=True,
    padding=True,
    max_length=max_length,
)

Wrap tokenized encodings into a custom PyTorch `Dataset` for use with HuggingFace `Trainer`.

In [59]:
class PoemDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

Initialize `PoemDataset` instances for train and test sets.

In [60]:
train_dataset = PoemDataset(train_encodings, y_train)
test_dataset = PoemDataset(test_encodings, y_test)

Configure training hyperparameters. Best checkpoint selected by weighted F1 on the evaluation set.

In [61]:
training_args = TrainingArguments(
    num_train_epochs= 3, # A small number of epochs        
    warmup_steps=25,
    per_device_train_batch_size= 8, #Small batch size for CPU
    per_device_eval_batch_size= 8,  # The same thing for evaluation
    learning_rate=5e-5,
    weight_decay=0.01, 
    output_dir="./results",
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="steps",
    save_strategy="best",
    metric_for_best_model="f1",
    load_best_model_at_end=True,
    use_cpu=True,                # Only change this if you know what you're doing.
    use_mps_device=False,        # Ditto 
)

`compute_metrics` returns weighted F1 — passed to `Trainer` for checkpoint selection.

In [62]:
def compute_metrics(pred):
    logits = pred.predictions
    labels = pred.label_ids
    preds = np.argmax(logits, axis=1)
    f1 = f1_score(labels, preds)
    return {"f1": f1}

Initialize the HuggingFace `Trainer`.

In [63]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
)

trainer = Trainer(
    model=model,                  # real model, not ...
    args=training_args,           # real TrainingArguments object
    train_dataset=train_dataset,  # your dataset
    eval_dataset=test_dataset,    # your dataset
    compute_metrics=compute_metrics,
)

Train the model. This takes several minutes on CPU.

In [64]:
# This will take several minutes to run.
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.674300,0.657437,0.000000
2,0.634600,0.647643,0.000000
3,0.621700,0.643107,0.000000


TrainOutput(global_step=30, training_loss=0.6435215632120769, metrics={'train_runtime': 117.7438, 'train_samples_per_second': 1.962, 'train_steps_per_second': 0.255, 'total_flos': 7661302032384.0, 'train_loss': 0.6435215632120769, 'epoch': 3.0})

Save the best model checkpoint for reuse in the poem generation evaluation step.

In [65]:
model_path = "results/best_minilm_classifier"
trainer.save_model(model_path)

### 2.3 LLM Classification via Ollama

Classify poems using `gemma3:1b-it-qat` served locally via Ollama with zero-shot and few-shot prompting. Results are compared against the baseline and the fine-tuned transformer.

In [66]:
ollama_model_name = "gemma3:1b-it-qat"

#### Zero-Shot Classification

`OllamaChatModel` wraps the Ollama OpenAI-compatible API into a simple callable, enabling modular swap-out of model backends.

In [67]:
class OllamaChatModel:
    def __init__(self, 
                 model_name: Optional[str] = None,
                 base_url: str = "http://localhost:11434/v1"):
        """
        Args:
            model_name: Name of a model that is accessible via Ollama.
                            Optional string.
            base_url:   String of the Url of your Ollama server. 
        """
        self.client = OpenAI(
            base_url=base_url,
            api_key="ollama"
        )
        self.model_name = model_name

    def __call__(self, 
                 prompt: str, 
                 model_name: Optional[str] = None, 
                 **kwargs) -> str:
        """
        Args:
            prompt:     Prompt to pass to the model. String.
            model_name: Name of a model that is accessible via Ollama.
                            If None, defaults to self.model_name. Optional string.
            kwargs:     Kwargs are passed directly to the OpenAI client's 
                            chat.completions.create method.
        Returns:
            response:   The model's response to the prompt as a string.
        """
        assert any((model_name, self.model_name)), "Please specify a model to use."
        if not model_name: 
            model = self.model_name
        
        
        response = self.client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            **kwargs,
        )
        return response.choices[0].message.content.strip()

Two zero-shot prompt templates are compared — one highly specific with constrained output format, one deliberately vague — to measure how prompt design affects accuracy and output validity.

In [68]:
zero_shot_prompt_specific = (
    "You are a classifier that labels poems as either 'natural' or 'other'.\n"
    "A poem is 'natural' if it is mainly about landscapes, seasons, weather, the moon, or other aspects of nature.\n"
    "Otherwise it is 'other'.\n"
    "Read the poem and respond with exactly one lowercase word: natural or other.\n\n"
    "Poem:\n{text}"
)

zero_shot_prompt_vague = (
    "Decide whether the following poem is mainly about nature or not.\n"
    "If it is about nature, answer natural; if it is not, answer other.\n"
    "Reply with a single lowercase word.\n\n"
    "Poem:\n{text}"
)

print("Specific prompt:\n", zero_shot_prompt_specific[:300], "...\n")
print("Vaguer prompt:\n", zero_shot_prompt_vague[:300], "...\n")

Specific prompt:
 You are a classifier that labels poems as either 'natural' or 'other'.
A poem is 'natural' if it is mainly about landscapes, seasons, weather, the moon, or other aspects of nature.
Otherwise it is 'other'.
Read the poem and respond with exactly one lowercase word: natural or other.

Poem:
{text} ...

Vaguer prompt:
 Decide whether the following poem is mainly about nature or not.
If it is about nature, answer natural; if it is not, answer other.
Reply with a single lowercase word.

Poem:
{text} ...



`llm_classify_zero_shot` runs inference over the test set, maps outputs to integer labels, and tracks output validity rate.

In [69]:
def llm_classify_zero_shot(model: OllamaChatModel, 
                           X: Sequence[str],
                           template: str,  
                           output_label_map: dict) -> Tuple[Sequence[int], Sequence[int]]: 
    """
    Args:
        model:            An OllamaChatModel. 
        X:                A sequence of strings with length N.
        template:         A prompt template (string with replacement field(s)).
        output_label_map: Dict with {output: integer_label}. Must contain an "other" key.
    Returns:
        preds:            Sequence of integer class predictions with length N.
        valids:           Sequence of validity values (0 for invalid output, or 1 for valid) with length N. 
    """
    
    preds = []
    valids = []

    # Normalize keys for matching
    output_map_norm = {k.lower(): v for k, v in output_label_map.items()}
    other_label = output_map_norm.get("other", 0)

    for text in X:
        # Be flexible about the replacement field name.
        prompt = template.format(text=text, poem=text, input=text)

        raw_response = model(prompt)
        response = raw_response.strip().lower()

        label = None
        valid = 0

        # Exact match first
        if response in output_map_norm:
            label = output_map_norm[response]
            valid = 1
        else:
            # Fallback: look for any key as a substring in the response
            for key, val in output_map_norm.items():
                if key in response:
                    label = val
                    valid = 1
                    break

        if label is None:
            # Could not map the response cleanly; treat as "other" and mark invalid
            label = other_label
            valid = 0

        preds.append(int(label))
        valids.append(int(valid))

    return preds, valids

Classify the test set with both prompt templates. Report precision, recall, F1, and output validity rate.

In [71]:
ollama_model = OllamaChatModel(model_name=ollama_model_name)

output_label_map = {
    "0": 0,
    "1": 1
}

zs_preds_specific, zs_valids_specific = llm_classify_zero_shot(
    ollama_model,
    X_test,
    zero_shot_prompt_specific,
    output_label_map,
)

zs_preds_vague, zs_valids_vague = llm_classify_zero_shot(
    ollama_model,
    X_test,
    zero_shot_prompt_vague,
    output_label_map,
)

#### Zero-Shot Results & Analysis

Between the two zero-shot prompts, the specific prompt did substantially better: higher F1 on the natural class and 100% valid outputs, which demonstrates that the model had much clearer understanding of the task when the instructions were clear, structured, and strongly constrained (“respond with exactly one word”). On the other hand, the vague prompt resulted in a collapse in behavior: nearly all the responses defaulted to “other,” and only 20% of them were valid with respect to the required format of output. This reveals how sensitive zero-shot prompting is to the design of a prompt: small differences in clarity and constraints can drastically alter the predictions.

Both zero-shot approaches performed significantly worse compared to the baseline logistic regression model. The baseline classifier makes use of bag-of-words features tailored to this dataset, so it learns domain-specific lexical cues that LLM does not automatically infer in a zero-shot setting. The fine-tuned transformer also outperformed both zero-shot prompts greatly, for fine-tuning adapts the model directly to the poem dataset and optimizes for the exact label distribution. Unlike zero-shot prompting, fine-tuning lets the model learn subtle distinctions in how nature imagery appears in classical Chinese poetry—something which prompting alone does not provide.

The performance metrics reinforce the observation made above with respect to prompt sensitivity and model capability. In the case of a particular zero-shot prompt, the model achieved an F1 score of 0.538 with a perfect valid output rate of 100%. It had great precision for class 0 at 1.00 but failed to recall most positive examples (recall of 0.08), which caused a generally low class F1 for that class. Class 1 was recognized well nonetheless, with a recall of 1.00, which showed that this constrained prompt helped to guide the model toward the target class effectively.

By contrast, the F1 for the model from the prompt lacking details was 0.000, though it yielded much higher accuracy at 0.65. It tagged everything as class 0, completely failing to identify any positive instances of class 1. Recall class 1: 0.00. Further, the valid output rate drops down to only 20%, which gives confirmation that even with minimal format expectations, it performs poorly if the prompt is not well-structured.

Interestingly, the fine-tuned transformer performed very poorly. Both the training and validation F1 scores were stuck at 0.000 throughout the steps of training, despite the training and validation loss going down gradually. This hints at either an inability of the model to learn from the training dataset or special difficulties presented by this dataset in terms of label distribution and structure, which this transformer architecture was not able to handle without further tuning or data augmentation.

The baseline model of logistic regression, by contrast, demonstrated strong and stable performance: a perfect F1 of 1.000 on the training set, and a solid 0.615 on the test set. Indeed, it captured class distinctions far more effectively than any of the zero-shot methods or the transformer—a fact which speaks to how simple, task-specific models can often outperform large language models when the latter are not properly adapted to a task.

Together, these results demonstrate that the success of an approach lies not only in model size or generality but in the level of alignment to the task, data-fit, and clarity of instructions provided. Overall, the differences in performance arise from the amount of task-specific learning each method can leverage. Zero-shot prompting is entirely based on the model's general world knowledge, which often is not enough to perform the narrow classification tasks and is highly sensitive to how formatting instructions are given. The baseline model captures simple statistical patterns present in the dataset, producing more stable results. The fine-tuned transformer benefits from both deep semantic representations and task-specific gradient updates, yielding the strongest and most reliable results

#### Few-Shot Classification

`FewShotPromptBuilder` is a reusable prompt templating framework that dynamically injects a configurable number of labeled examples into a prompt at inference time.

In [74]:
class FewShotPromptBuilder:
    def __init__(self, prompt_template, example_template, example_data):
        import re
        self.prompt_fields = set(self.extract_fields(prompt_template))
        assert "examples" in self.prompt_fields, \
            "prompt_template must include an {examples} replacement field"

        self.prompt_template = prompt_template

        self.validate_example_data(example_data)
        self.example_data = example_data
        self.num_examples = len(example_data)
        self.example_length = len(example_data[0])

        self.validate_example_template(example_template)
        self.example_template = example_template

    def __call__(self, n, random=False, **kwargs):
        import numpy as np

        assert n <= self.num_examples, \
            f"{n} is greater than total examples ({self.num_examples})"

        assert all(k in self.prompt_fields for k in kwargs), \
            "Mismatch between kwargs and prompt_template fields"

        if random:
            indices = np.random.choice(self.num_examples, size=n, replace=False)
        else:
            indices = np.arange(n)

        sampled_examples = [self.example_data[i] for i in indices]
        examples_string = self.format_examples(sampled_examples)

        all_kwargs = dict(kwargs)
        all_kwargs["examples"] = examples_string

        return self.prompt_template.format(**all_kwargs)

    def format_examples(self, sampled_examples):
        formatted = [
            self.example_template.format(*example)
            for example in sampled_examples
        ]
        return "\n\n".join(formatted)

    def validate_example_data(self, example_data):
        length = len(example_data[0])
        for i, example in enumerate(example_data):
            assert len(example) == length, \
                f"Example {i} has wrong length"

    def validate_example_template(self, example_template):
        fields = [int(i) for i in self.extract_fields(example_template)]
        if len(fields) > 1:
            assert max(fields) == self.example_length - 1
            assert len(fields) == self.example_length
            assert set(fields) == set(range(self.example_length))
        else:
            assert fields[0] == 0

    @staticmethod
    def extract_fields(template):
        import re
        pattern = re.compile(r"\{(\w+)}")
        return re.findall(pattern, template)

Verify `FewShotPromptBuilder` initializes and formats prompts correctly.

In [75]:
FewShotPromptBuilder(
    "Here is a formatted example: \n{examples}\nThese are some other kwargs that can be formatted:\n{item}\n{other_stuff}", 
    "Here's text: {0}\nand a label: {1}\n", 
    [("text", 1), ("other_text", 2)]
)

Curate balanced examples (5 natural + 5 other) from the training set for few-shot prompts.

In [76]:
label_output_map = {1: "NATURAL", 0: "OTHER"}
output_label_map = {"natural": 1, "other": 0}

natural_idx = np.argwhere(y_train).squeeze()[:5]
other_idx = np.argwhere(np.abs(np.array(y_train) - 1)).squeeze()[:5]

balanced_idx = np.concat((natural_idx, other_idx))
example_labels = np.concat((np.ones(5), np.zeros(5))).astype(int)
shuffle_idx = np.random.permutation(np.arange(balanced_idx.shape[0]))

balanced_idx = balanced_idx[shuffle_idx].tolist()
example_labels = example_labels[shuffle_idx].tolist()

balanced_examples = [X_train[i] for i in balanced_idx]


poem_example_data = [(text, label_output_map[label]) 
                     for text, label in zip(balanced_examples, example_labels)]

Define prompt and example templates, then initialize a `FewShotPromptBuilder` with the curated examples.

In [77]:
# Your prompts and code here
few_shot_prompt_template = (
    "You are a helpful assistant that classifies poems as NATURAL or OTHER.\n"
    "Use the examples below to learn the mapping from poems to labels.\n\n"
    "{examples}\n\n"
    "Now classify the following poem.\n"
    "Respond with exactly one word, either NATURAL or OTHER.\n\n"
    "Poem:\n{poem}"
)

few_shot_example_template = (
    "Poem:\n{0}\n"
    "Label: {1}"
)

few_shot_builder = FewShotPromptBuilder(
    prompt_template=few_shot_prompt_template,
    example_template=few_shot_example_template,
    example_data=poem_example_data,
)

print(few_shot_builder(2, poem="<<example poem here>>"))


You are a helpful assistant that classifies poems as NATURAL or OTHER.
Use the examples below to learn the mapping from poems to labels.

Poem:
Army Life

Clouds on frontier have darkened mountains clad in snow; The town with Gate of Jade stands far away, forlorn. We will not leave the desert till we beat the foe, Although in war our golden armour be outworn.
Label: OTHER

Poem:
Mount Heaven's Gate Viewed from Afar

Breaking Mount Heaven's Gate, the great River rolls through; Green billows eastward flow and here turn to the north. From both sides of the River thrust out the cliffs blue; Leaving the sun behind, a lonely sail comes forth.
Label: NATURAL

Now classify the following poem.
Respond with exactly one word, either NATURAL or OTHER.

Poem:
<<example poem here>>


`llm_classify_few_shot` extends zero-shot classification with dynamic example injection, supporting fixed or random example sampling.

In [79]:
def llm_classify_few_shot(model: OllamaChatModel, 
                          X: Sequence[str],
                          prompt_builder: FewShotPromptBuilder, 
                          output_label_map: dict,
                          num_examples: int = 5,
                          sample_random: bool = False) -> Tuple[Sequence[int], Sequence[int]]:
    """
    Args:
        model:            An OllamaChatModel. 
        X:                A sequence of strings with length N.
        prompt_builder:   A FewShotPromptBuilder.
        template:         A prompt template (string with replacement field(s)).
        output_label_map: Dict with {output: integer_label}. Must contain an "other" key.
        num_examples:     Number of examples to sample with the FewShotPromptBuilder. 
        sample_random:    Whether to sample randomly with the FewShotPromptBuilder. Default is False.
    Returns:
        preds:            Sequence of integer class predictions with length N.
        valids:           Sequence of validity values (0 for invalid output, 1 for valid) with length N. 
    """
    preds = []
    valids = []

    output_map_norm = {k.lower(): v for k, v in output_label_map.items()}
    other_label = output_map_norm.get("other", 0)

    for text in X:
        prompt = few_shot_builder(
            num_examples,
            random=sample_random,
            poem= text,
        )

        raw_response = model(prompt)
        response = raw_response.strip().lower()

        label = None
        valid = 0

        if response in output_map_norm:
            label = output_map_norm[response]
            valid = 1
        else:
            for key, val in output_map_norm.items():
                if key in response:
                    label = val
                    valid = 1
                    break

        if label is None:
            label = other_label
            valid = 0

        preds.append(int(label))
        valids.append(int(valid))

    return preds, valids


Benchmark few-shot classification across `num_examples` ∈ {1, 3, 5} and report results as a summary DataFrame.

In [81]:
few_shot_results = []

for k in [1, 3, 5]:
    fs_preds, fs_valids = llm_classify_few_shot(
        ollama_model,
        X_test,
        few_shot_builder,
        output_label_map,
        num_examples=k,
        sample_random=True,
    )

    f1 = f1_score(y_test, fs_preds)
    valid_rate = np.mean(fs_valids)

    few_shot_results.append(
        {
            "num_examples": k,
            "f1": f1,
            "valid_rate": valid_rate,
        }
    )

few_shot_results_df = pd.DataFrame(few_shot_results)

In [82]:
# Print results here
print(few_shot_results_df)

   num_examples   f1  valid_rate
0             1  0.0         1.0
1             3  0.0         1.0
2             5  0.0         1.0


#### Few-Shot Results & Analysis

What's really stood out to me across the entire classification process is how dramatically the different methods seemed to embody different kinds of "intelligence," and how poorly those intuitions transfer between approaches. The zero-shot and few-shot prompting failures revealed something important: LLMs are not classifiers, and without either fine-tuning or extremely careful prompting, they default to shallow heuristics, semantic priors, and output regularities that often have nothing to do with the underlying literary distinctions in the dataset. I was not surprised that the fine-tuned transformer and the even simpler logistic-regression baseline outperformed the LLM-because those models were explicitly trained to notice the statistical cues that distinguish "nature" poems from "other" poems-yet I was struck by how decisively the LLM ignored the structure of the few-shot examples. Even with balanced demonstrations, the model collapsed onto one label, suggesting that small models served through Ollama are not performing genuine in-context learning but instead matching surface patterns or the most salient recurrent token in the examples. If I wanted to improve performance, I would tighten the output constraints even more, strip all fluff from the prompt, expose the label schema more prominently, and experiment with formatting that forces the model into a rigid pattern-completion mode rather than an instruction-following mode. My prompting process was not entirely "vibes-only," but it was iterative: I would generate a batch of predictions, inspect the invalid or off-format responses, adjust the phrasing to reduce ambiguity, and re-run until the valid-output rate stabilized; however, the classifier's fundamental behavior was remarkably stubborn, which reinforced the idea that prompting alone cannot replace model capacity, training, or proper supervision. Adding to this, the empirical results made the limitations even clearer: whether using 1, 3, or 5 examples, the F1 score remained at 0.0 while the valid output rate stayed consistently at 1.0. This confirmed that though the model followed formatting instructions reliably, it utterly failed to learn or generalize the classification task. The consistent F1 score of zero across increasing few-shot examples underscores that no meaningful signal was being extracted from the demonstrations—further evidence that surface-level pattern recognition was at play, not true in-context learning

## 3. Poem Generation

Generate synthetic poems in two categories — **NATURAL** and **OTHER** — using few-shot prompting with style constraints: short lines, elemental imagery, no rhyme, no modern language.

Two `FewShotPromptBuilder` instances are initialized from the training set — one with nature examples, one with other examples — to condition the model on each category separately.

In [83]:
gen_prompt_template = (
    "You are a poet that writes extremely short classical Chinese-style poems.\n"
    "Your poems MUST:\n"
    "- be 1–3 short lines\n"
    "- be highly concise\n"
    "- use elemental imagery (mountains, rivers, wind, frost)\n"
    "- avoid modern language, story structure, and explanation\n"
    "- avoid metaphors common in English poetry\n"
    "- avoid rhyming\n"
    "- match the STYLE of the following labeled examples.\n\n"
    "{examples}\n\n"
    "Now write a NEW poem labeled: {label}.\n"
    "ONLY output the poem itself. No explanation, no commentary."
)

gen_example_template = (
    "Poem:\n{0}\n"
    "Label: {1}"
)

gen_builder = FewShotPromptBuilder(
    prompt_template=gen_prompt_template,
    example_template=gen_example_template,
    example_data=poem_example_data
)

ollama_gen = OllamaChatModel(model_name=ollama_model_name)

### 3.1 Generation

`generate_poems` samples `num_examples` from the builder and calls the LLM `num_poems` times, returning generated texts.

In [84]:
def generate_poems(model: OllamaChatModel, 
                   prompt_builder: FewShotPromptBuilder, 
                   num_poems: int, 
                   num_examples: int, 
                   random_sample: bool = False,
                   label: str = "NATURAL") -> Sequence[str]:

    poems = []
    for _ in range(num_poems):
        prompt = prompt_builder(
            num_examples,
            random=random_sample,
            label=label
        )
        text = model(prompt)
        poems.append(text.strip())
    return poems

In [85]:
# Generate poems
num_poems_per_class = 10
num_examples_per_prompt = 3

natural_poems = generate_poems(
    ollama_gen,
    gen_builder,
    num_poems=num_poems_per_class,
    num_examples=num_examples_per_prompt,
    random_sample=True,
    label="NATURAL"
)

other_poems = generate_poems(
    ollama_gen,
    gen_builder,
    num_poems=num_poems_per_class,
    num_examples=num_examples_per_prompt,
    random_sample=True,
    label="OTHER"
)

generated_poems = natural_poems + other_poems
generated_labels = [1] * len(natural_poems) + [0] * len(other_poems)



Print generated poems for qualitative inspection.

In [86]:
print("=== NATURAL POEMS ===")
for i, poem in enumerate(natural_poems, start=1):
    print(f"--- Natural poem {i} ---")
    print(poem)
    print()

=== NATURAL POEMS ===
--- Natural poem 1 ---
山河静默。

--- Natural poem 2 ---
Mountain white peaks rise.
River whispers low.
Wind sighs through pines.

--- Natural poem 3 ---
River's breath turns grey.

--- Natural poem 4 ---
East Wind whispers through pines.
Sunlight falls on gray stone.
Clear stream rushes low.

--- Natural poem 5 ---
Mountains rise.
River flows.
Wind whispers.

--- Natural poem 6 ---
River’s breath flows still.
Wind whispers stone.
Frost clings high.

--- Natural poem 7 ---
River flows onward.
Wind whispers through pines.
Frost blooms on stone.

--- Natural poem 8 ---
Mountains breathe air.
River flows downwards.
Frost clings to stone.

--- Natural poem 9 ---
Mountain Stream at Dusk

--- Natural poem 10 ---
White frost clings tight.
River whispers deep.
Stone breathes the wind.



In [87]:
print("=== OTHER POEMS ===")
for i, poem in enumerate(other_poems, start=1):
    print(f"--- Other poem {i} ---")
    print(poem)
    print()

=== OTHER POEMS ===
--- Other poem 1 ---
Iceflower Mist

--- Other poem 2 ---
Snowdrift falls

--- Other poem 3 ---
Dragon’s Breath

--- Other poem 4 ---
Snowdrift Silence

--- Other poem 5 ---
Wind’s frozen breath

--- Other poem 6 ---


--- Other poem 7 ---
Wind whispers.
Stone sleeps.
Grey.

--- Other poem 8 ---
Shadows lengthen.

--- Other poem 9 ---


--- Other poem 10 ---
冰水凝结
风蚀河落



### 3.2 Evaluating Generated Poems with the Fine-Tuned Classifier

Use the fine-tuned `all-MiniLM-L6-v2` classifier to predict labels for generated poems. This creates a generative evaluation loop: the classifier scores how well the LLM captured the stylistic distinction between nature and other poems through prompting alone.

Tokenize generated poems using the same tokenizer as the fine-tuned classifier.

In [88]:
gen_encodings = tokenizer(
    generated_poems,
    truncation=True,
    padding=True,
    max_length=max_length,
)

gen_encodings = {k: torch.tensor(v) for k, v in gen_encodings.items()}

Load the saved fine-tuned model and run inference on the generated poems.

Load your fine-tuned transformer model (if you `del`ed it earlier), then get predictions from the model for each of your generated poems. 

In [89]:
clf_model = AutoModelForSequenceClassification.from_pretrained(model_path)

clf_model.eval()
with torch.no_grad():
    logits = clf_model(**gen_encodings).logits
preds = logits.argmax(dim=1).cpu().numpy()

Report precision, recall, accuracy, and F1 for predictions on the generated poems.

In [90]:
precision = precision_score(generated_labels, preds)
recall = recall_score(generated_labels, preds)
accuracy = accuracy_score(generated_labels, preds)
f1 = f1_score(generated_labels, preds)

print("Precision:", precision)
print("Recall:", recall)
print("Accuracy:", accuracy)
print("F1:", f1)

Precision: 0.0
Recall: 0.0
Accuracy: 0.5
F1: 0.0


C:\Users\akinw\miniconda3\envs\3350\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## 4. Saving Results

In [92]:
import os
os.makedirs("results", exist_ok=True)

few_shot_results_df.to_csv("results/few_shot_results.csv", index=False)

with open("results/generated_poems.txt", "w", encoding="utf-8") as f:
    for i, poem in enumerate(natural_poems, 1):
        f.write(f"--- Natural {i} ---\n{poem}\n\n")
    for i, poem in enumerate(other_poems, 1):
        f.write(f"--- Other {i} ---\n{poem}\n\n")